# Agentic AI Systems — Worklet 15
# Executive Decision Support System
## Lab 3 · Skill Creation — reusable plan & format skills

**Phase A · Foundations** &nbsp;|&nbsp; **Goal:** package repeatable capability as reusable, schema-typed skills.

---

### What Lab 3 contributes to Worklet 15

| | |
|---|---|
| **Inputs** | Parsed inputs; template spec. |
| **Outputs** | A locked plan object; a formatted decision-brief shell. |
| **Agents / nodes** | Planning and assembly emerge as **skills**. |
| **Deliverable** | Two callable skills with schemas, invoked by the agent. |
| **Acceptance** | **The same skill works for two different cases without code changes.** |
| **Pitfall to avoid** | Keep skills case-agnostic and typed; hard-coding one case breaks reuse. |

> Chapter 15 pseudocode:
> ```
> skill plan(inputs) -> Plan:  return expand(inputs).lock()
> skill format_decision(result, template) -> Document
> ```

### Where Labs 1–2 left off

Lab 1 built the coordinator loop. Lab 2 built the typed tool layer that produces every figure. Both were *procedures written into the notebook*. Lab 3 packages the repeatable parts as **skills**: instructions + constraints + typed input/output contracts + the exact set of tools each is allowed to call.

### The two skills

| Skill | What it does | Tools it may call |
|---|---|---|
| `review_domain` | Reviews one domain's approved KPIs against target, with per-department evidence | 5 read/compute tools |
| `format_document` | Renders any titled, sectioned payload as a leadership document | **none** — pure layout |

The domain is an **input**, not a branch. Cell 5 runs `review_domain` against five different cases; Cell 6 feeds `format_document` a KPI review and an unrelated logistics memo. Cell 9 proves nothing is hard-coded by adding a KPI to the registry **at run time** and showing the same skill object picks it up.

### How to run

Runs top to bottom with **no API key** — this lab is about capability packaging, which is deterministic. Cell 1 writes the shared Labs 1–2 foundation to `edss_core.py`; Cell 9 executes the rubric as assertions.

---

### Cell 1 · The shared foundation

The Labs 1–2 engine, unchanged: the institutional data, the typed tool contracts, the `ToolRouter` choke point, the `Connector`, and the validation gate. Written to `edss_core.py` so this notebook runs standalone in Colab, and so Labs 3–8 all build on the same tools and the same numbers.

*(Collapse this cell — it is prior work, restated only so the notebook is self-contained.)*

In [1]:
%%writefile edss_core.py
"""edss_core — shared foundation for the Worklet 15 lab notebooks.

Worklet 15 · Executive Decision Support System · Agentic AI (26ECAC401)

This is the Lab 1 + Lab 2 foundation, unchanged: the institutional data, the
typed tool contracts, the single ToolRouter choke point, the Connector, and the
validation gate. Labs 3-8 build on top of it rather than restating it, so every
notebook works against the same tools and the same numbers.

Written to disk by the first code cell of each lab notebook, so each notebook
runs standalone in Colab with no setup.
"""

from __future__ import annotations

import csv
import io
import json
import math
import re
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

# 1 · Institutional data (identical to Lab 2, Cell 3)
# =============================================================================

REPORTING_PERIOD = "academic year 2025-26"

DEPARTMENT_REPORTS_CSV = """dept_code,dept_name,category,students_appeared,students_passed,co_po_attainment,sanctioned_faculty,publications,citations,grants_lakh,eligible_students,placed_students,total_package_lpa,interns_completed,budget_allocated_lakh,budget_spent_lakh,figure_status
CSE,Computer Science & Engineering,circuit,240,231,2.71,34,58,402,96.5,228,212,2247.2,205,420,393,final
ECE,Electronics & Communication,circuit,180,168,2.58,26,34,221,41.0,172,143,1058.2,149,310,276,final
ISE,Information Science & Engineering,circuit,120,116,2.66,18,21,140,22.5,114,103,947.6,96,240,221,final
EEE,Electrical & Electronics,circuit,120,109,2.41,17,16,88,18.0,116,86,524.6,84,210,181,final
MECH,Mechanical Engineering,core,180,158,2.33,24,19,95,27.5,174,111,599.4,102,300,247,final
CIVIL,Civil Engineering,core,120,101,2.19,16,11,47,12.0,118,62,285.2,61,190,149,final
CHEM,Chemical Engineering,core,60,54,2.28,11,14,76,15.5,58,33,168.3,34,140,118,final
BT,Biotechnology,core,60,55,2.35,10,12,63,9.5,57,29,142.1,30,130,102,final
"""

NUMERIC_COLUMNS = {
    "students_appeared": int, "students_passed": int, "co_po_attainment": float,
    "sanctioned_faculty": int, "publications": int, "citations": int,
    "grants_lakh": float, "eligible_students": int, "placed_students": int,
    "total_package_lpa": float, "interns_completed": int,
    "budget_allocated_lakh": float, "budget_spent_lakh": float,
}

KPI_DEFINITIONS: Dict[str, Dict[str, Any]] = {
    "KPI-A1": {"name": "Institution pass percentage", "domain": "Academic",
               "method": "ratio", "numerator": "students_passed",
               "denominator": "students_appeared", "scale": 100, "unit": "%",
               "formula": "sum(students_passed) / sum(students_appeared) x 100",
               "target": 92.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Academic ERP"},
    "KPI-A2": {"name": "CO/PO attainment index", "domain": "Academic",
               "method": "weighted_mean", "value_field": "co_po_attainment",
               "weight_field": "students_appeared", "unit": "index 0-3",
               "formula": "sum(co_po_attainment x students_appeared) / sum(students_appeared)",
               "target": 2.50, "target_source": "NAAC criterion 2",
               "direction": "higher_is_better", "source_system": "Academic ERP (OBE module)"},
    "KPI-R1": {"name": "Publications per faculty", "domain": "Research",
               "method": "ratio", "numerator": "publications",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(publications) / sum(sanctioned_faculty)",
               "target": 1.50, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R2": {"name": "Citations per faculty", "domain": "Research",
               "method": "ratio", "numerator": "citations",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(citations) / sum(sanctioned_faculty)",
               "target": 9.00, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R3": {"name": "Grants sanctioned", "domain": "Research",
               "method": "sum", "value_field": "grants_lakh", "unit": "INR lakh",
               "formula": "sum(grants_lakh)",
               "target": 300.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-P1": {"name": "Placement percentage", "domain": "Placement",
               "method": "ratio", "numerator": "placed_students",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(placed_students) / sum(eligible_students) x 100",
               "target": 80.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P2": {"name": "Average package", "domain": "Placement",
               "method": "ratio", "numerator": "total_package_lpa",
               "denominator": "placed_students", "scale": 1, "unit": "LPA",
               "formula": "sum(total_package_lpa) / sum(placed_students)",
               "target": 8.00, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P3": {"name": "Internship completion", "domain": "Placement",
               "method": "ratio", "numerator": "interns_completed",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(interns_completed) / sum(eligible_students) x 100",
               "target": 85.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-F1": {"name": "Budget utilisation", "domain": "Finance",
               "method": "ratio", "numerator": "budget_spent_lakh",
               "denominator": "budget_allocated_lakh", "scale": 100, "unit": "%",
               "formula": "sum(budget_spent_lakh) / sum(budget_allocated_lakh) x 100",
               "target": 90.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Finance & Accounts ledger",
               "figure_status": "provisional",
               "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes "
                         "books in August."},
}

BENCHMARK_API_DB = {
    "KPI-R1": {"peer_median": 1.82, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-R2": {"peer_median": 11.40, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-P1": {"peer_median": 78.60, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
}

DOMAINS = ["Academic", "Research", "Placement", "Finance"]
DOMAIN_KPIS = {d: [k for k, v in KPI_DEFINITIONS.items() if v["domain"] == d] for d in DOMAINS}

# Registrar-authorised read access, per Gate 1 of the hybrid execution methodology.
ACCESS_REGISTER = {
    "Academic":  {"source_system": "Academic ERP — examination & OBE records",
                  "read_access": "granted by Registrar on 2026-06-01", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["pass percentage", "CO/PO attainment", "curriculum compliance"]},
    "Research":  {"source_system": "Research & Publications repository (Scopus-synced)",
                  "read_access": "granted by Registrar on 2026-06-01", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["publications", "citations", "grants sanctioned"]},
    "Placement": {"source_system": "Placement Cell tracker",
                  "read_access": "granted by Registrar on 2026-06-04", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["placement percentage", "average package",
                              "internship completion"],
                  "caveat": "3 provisional offers are excluded until the Placement Cell "
                            "verifies them."},
    "Finance":   {"source_system": "Finance & Accounts ledger",
                  "read_access": "granted by Registrar on 2026-06-07",
                  "figure_status": "provisional",
                  # 2025-26 is deliberately absent: the books are not closed, and the
                  # Lab 1 planner must notice that rather than plan over it.
                  "periods": ["2023-24", "2024-25"],
                  "metrics": ["budget utilisation", "revenue", "grant funds"],
                  "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes "
                            "books in August."},
}


# =============================================================================
# 2 · Tool contracts (identical to Lab 2, Cell 4)
# =============================================================================

TOOL_SCHEMAS: Dict[str, Dict[str, Any]] = {
    "read_department_reports": {
        "description": "Read per-department metric rows for one reporting period.",
        "parameters": {"type": "object", "properties": {
            "period": {"type": "string"},
            "departments": {"type": "array", "items": {"type": "string"}},
            "category": {"type": "string", "enum": ["circuit", "core"]}},
            "required": ["period"]},
        "returns": "{rows, row_count, source_system} | {error}"},
    "read_kpi_definitions": {
        "description": "Read approved KPI definitions: formula, unit, target, authority.",
        "parameters": {"type": "object", "properties": {
            "kpi_ids": {"type": "array", "items": {"type": "string"}}}, "required": []},
        "returns": "{definitions, count} | {error}"},
    "read_external_benchmark": {
        "description": "Published peer median for a KPI. Coverage is partial.",
        "parameters": {"type": "object", "properties": {"kpi_id": {"type": "string"}},
                       "required": ["kpi_id"]},
        "returns": "{kpi_id, peer_median, cohort, year} | {error}"},
    "filter_rows": {
        "description": "Filter department rows by an exact field value; errors on empty.",
        "parameters": {"type": "object", "properties": {
            "rows": {"type": "array", "items": {"type": "object"}},
            "field": {"type": "string"}, "value": {"type": "string"}},
            "required": ["rows", "field", "value"]},
        "returns": "{rows, row_count, filter} | {error}"},
    "compute_kpi": {
        "description": "Compute one approved KPI over department rows, with evidence.",
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string"},
            "rows": {"type": "array", "items": {"type": "object"}}},
            "required": ["kpi_id", "rows"]},
        "returns": "{kpi_id, value, unit, formula, evidence, source_system} | {error}"},
    "compare_to_target": {
        "description": "Compare a KPI value to its approved target; flags rounding risk.",
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string"}, "value": {"type": "number"}},
            "required": ["kpi_id", "value"]},
        "returns": "{value, target, variance, status, rounding_warning} | {error}"},
    "compute_variance": {
        "description": "Difference between a KPI value and any reference figure.",
        "parameters": {"type": "object", "properties": {
            "value": {"type": "number"}, "reference": {"type": "number"},
            "label": {"type": "string"}}, "required": ["value", "reference", "label"]},
        "returns": "{variance, abs_variance, pct_of_reference} | {error}"},
    "draft_notification": {
        "description": "DRAFT a leadership notification. Never sends. Sources required.",
        "parameters": {"type": "object", "properties": {
            "audience": {"type": "string"}, "subject": {"type": "string"},
            "findings": {"type": "array", "items": {"type": "object"}}},
            "required": ["audience", "subject", "findings"]},
        "returns": "{status: 'DRAFT - NOT SENT', body, requires_approval_from} | {error}"},
}


def _typed_rows(text: str) -> List[Dict[str, Any]]:
    rows = []
    for raw in csv.DictReader(io.StringIO(text)):
        row = dict(raw)
        for col, caster in NUMERIC_COLUMNS.items():
            if col in row:
                try:
                    row[col] = caster(row[col])
                except (TypeError, ValueError):
                    return [{"error": f"column '{col}' is not {caster.__name__}"}]
        rows.append(row)
    return rows


# Optional demo latency, so the parallel/sequential distinction in the run
# timeline is visible. Off by default; never affects any computed figure.
SOURCE_LATENCY_SECONDS = 0.0


def read_department_reports(period: str, departments=None, category=None) -> Dict[str, Any]:
    if period != REPORTING_PERIOD:
        return {"error": f"no department report export for '{period}'; the only loaded "
                         f"period is '{REPORTING_PERIOD}'"}
    if SOURCE_LATENCY_SECONDS:
        time.sleep(SOURCE_LATENCY_SECONDS)
    rows = _typed_rows(DEPARTMENT_REPORTS_CSV)
    if rows and "error" in rows[0]:
        return {"error": rows[0]["error"]}
    if departments:
        known = {r["dept_code"] for r in rows}
        unknown = [d for d in departments if d not in known]
        if unknown:
            return {"error": f"unknown department code(s): {unknown}; known: {sorted(known)}"}
        rows = [r for r in rows if r["dept_code"] in departments]
    if category:
        rows = [r for r in rows if r["category"] == category]
        if not rows:
            return {"error": f"no departments in category '{category}'"}
    if not rows:
        return {"error": "empty department report export"}
    return {"rows": rows, "row_count": len(rows), "period": period,
            "source_system": "Department reports export (Registrar-authorised read)"}


def read_kpi_definitions(kpi_ids=None) -> Dict[str, Any]:
    if kpi_ids:
        unknown = [k for k in kpi_ids if k not in KPI_DEFINITIONS]
        if unknown:
            return {"error": f"unknown KPI id(s): {unknown}"}
        defs = {k: KPI_DEFINITIONS[k] for k in kpi_ids}
    else:
        defs = dict(KPI_DEFINITIONS)
    return {"definitions": defs, "count": len(defs),
            "source_system": "Institutional KPI register (approved definitions)"}


def read_external_benchmark(kpi_id: str) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    record = BENCHMARK_API_DB.get(kpi_id)
    if record is None:
        return {"error": f"no published external benchmark for {kpi_id} "
                         f"({KPI_DEFINITIONS[kpi_id]['name']}) in the benchmark API"}
    return {"kpi_id": kpi_id, **record, "source_system": "External benchmark API (NIRF cohort)"}


def filter_rows(rows, field: str, value: str) -> Dict[str, Any]:
    if not rows:
        return {"error": "filter_rows received an empty row set"}
    if field not in rows[0]:
        return {"error": f"unknown field '{field}'; available: {sorted(rows[0])}"}
    subset = [r for r in rows if str(r[field]) == str(value)]
    if not subset:
        return {"error": f"filter {field}={value!r} matched 0 of {len(rows)} rows — "
                         f"refusing to return an empty subset"}
    return {"rows": subset, "row_count": len(subset), "filter": f"{field}=={value}",
            "filtered_from": len(rows)}


def compute_kpi(kpi_id: str, rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Generic: the arithmetic comes from the KPI definition, not from this code."""
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    if not rows:
        return {"error": f"cannot compute {kpi_id} over an empty row set"}
    d = KPI_DEFINITIONS[kpi_id]
    method, evidence = d["method"], []
    try:
        if method == "ratio":
            num_f, den_f, scale = d["numerator"], d["denominator"], d["scale"]
            num = sum(r[num_f] for r in rows)
            den = sum(r[den_f] for r in rows)
            if den == 0:
                return {"error": f"{kpi_id}: denominator sum({den_f}) is zero"}
            value = num / den * scale
            inputs = {f"sum({num_f})": round(num, 4), f"sum({den_f})": round(den, 4)}
            for r in rows:
                dv = (r[num_f] / r[den_f] * scale) if r[den_f] else None
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[num_f], "denominator": r[den_f],
                                 "numerator_field": num_f, "denominator_field": den_f,
                                 "dept_value": round(dv, 2) if dv is not None else None,
                                 "share_of_denominator_pct": round(r[den_f] / den * 100, 2)})
        elif method == "weighted_mean":
            vf, wf = d["value_field"], d["weight_field"]
            wsum = sum(r[wf] for r in rows)
            if wsum == 0:
                return {"error": f"{kpi_id}: total weight sum({wf}) is zero"}
            value = sum(r[vf] * r[wf] for r in rows) / wsum
            inputs = {f"sum({vf} x {wf})": round(sum(r[vf] * r[wf] for r in rows), 4),
                      f"sum({wf})": wsum}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[vf], "denominator": r[wf],
                                 "numerator_field": vf, "denominator_field": wf,
                                 "dept_value": r[vf],
                                 "share_of_denominator_pct": round(r[wf] / wsum * 100, 2)})
        elif method == "sum":
            vf = d["value_field"]
            value = sum(r[vf] for r in rows)
            inputs = {f"sum({vf})": round(value, 4)}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[vf], "denominator": None,
                                 "numerator_field": vf, "denominator_field": None,
                                 "dept_value": r[vf],
                                 "share_of_denominator_pct": round(r[vf] / value * 100, 2)
                                 if value else None})
        else:
            return {"error": f"unsupported KPI method '{method}' for {kpi_id}"}
    except KeyError as missing:
        return {"error": f"{kpi_id}: department rows are missing column {missing}"}

    statuses = {r.get("figure_status", "final") for r in rows}
    figure_status = d.get("figure_status") or ("final" if statuses == {"final"} else "mixed")
    out = {"kpi_id": kpi_id, "name": d["name"], "domain": d["domain"],
           "value": round(value, 4), "unit": d["unit"], "formula": d["formula"],
           "method": method, "n_departments": len(rows),
           "departments": [r["dept_code"] for r in rows],
           "inputs_used": inputs, "evidence": evidence,
           "source_system": d["source_system"], "figure_status": figure_status}
    if d.get("caveat"):
        out["caveat"] = d["caveat"]
    return out


def compare_to_target(kpi_id: str, value: float) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    d = KPI_DEFINITIONS[kpi_id]
    target = d["target"]
    variance = value - target
    pct_of_target = value / target * 100 if target else None
    gap_pct = abs(variance) / target * 100 if target else 0
    status = "on target" if variance >= 0 else ("at risk" if gap_pct <= 2.0 else "below target")
    rounding_warning = None
    if variance < 0:
        for places in (0, 1):
            if round(value, places) >= target:
                rounding_warning = (f"{value:.4g} rounds to {round(value, places)} at "
                                    f"{places} decimal place(s), which would read as meeting "
                                    f"the target of {target}. Report the unrounded figure.")
                break
    return {"kpi_id": kpi_id, "name": d["name"], "value": round(value, 4), "target": target,
            "target_source": d["target_source"], "unit": d["unit"],
            "variance": round(variance, 4),
            "pct_of_target": round(pct_of_target, 2) if pct_of_target else None,
            "status": status, "rounding_warning": rounding_warning}


def compute_variance(value: float, reference: float, label: str) -> Dict[str, Any]:
    if reference == 0:
        return {"error": f"cannot compute variance against a zero reference ({label})"}
    variance = value - reference
    return {"value": round(value, 4), "reference": round(reference, 4), "label": label,
            "variance": round(variance, 4), "abs_variance": round(abs(variance), 4),
            "pct_of_reference": round(value / reference * 100, 2)}


def draft_notification(audience: str, subject: str, findings: List[Dict[str, Any]]):
    if not findings:
        return {"error": "refusing to draft a notification with no findings"}
    unsourced = [f for f in findings if not f.get("source")]
    if unsourced:
        return {"error": f"{len(unsourced)} finding(s) carry no source; every figure in a "
                         f"leadership notification must cite where it came from"}
    body = "\n".join(f"  {i}. {f['statement']}\n     source: {f['source']}"
                     for i, f in enumerate(findings, 1))
    return {"status": "DRAFT - NOT SENT", "audience": audience, "subject": subject,
            "body": body, "findings_cited": len(findings),
            "requires_approval_from": "Principal / Registrar (Chapter 15 human gate)",
            "note": "This tool cannot send. Circulation happens only after human approval."}


TOOLS = {
    "read_department_reports": read_department_reports,
    "read_kpi_definitions": read_kpi_definitions,
    "read_external_benchmark": read_external_benchmark,
    "filter_rows": filter_rows,
    "compute_kpi": compute_kpi,
    "compare_to_target": compare_to_target,
    "compute_variance": compute_variance,
    "draft_notification": draft_notification,
}


# =============================================================================
# 3 · Router, connector, validation gate (Lab 2, Cells 5-6)
# =============================================================================

class ToolContractError(Exception):
    """Raised before a call when arguments do not satisfy the tool schema."""


class ToolError(Exception):
    """Raised after a call when the tool returned an explicit error."""


_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "array": list, "object": dict}


def _summarise(obj, _depth=0):
    if isinstance(obj, list):
        if len(obj) > 3 and all(isinstance(x, dict) for x in obj):
            return f"[{len(obj)} rows: {', '.join(str(x.get('dept_code', '?')) for x in obj)}]"
        return [_summarise(x, _depth + 1) for x in obj]
    if isinstance(obj, dict):
        return {k: _summarise(v, _depth + 1) for k, v in obj.items()}
    if isinstance(obj, str) and len(obj) > 160:
        return obj[:157] + "..."
    return obj


class ToolRouter:
    """The single choke point. Validates, executes, logs, raises on error."""

    def __init__(self, agent: str = "Coordinator"):
        self.agent = agent
        self.trace: List[Dict[str, Any]] = []
        self.raw_outputs: List[Any] = []

    def validate_input(self, tool: str, kwargs: Dict[str, Any]) -> None:
        if tool not in TOOL_SCHEMAS:
            raise ToolContractError(f"no such tool: '{tool}'")
        schema = TOOL_SCHEMAS[tool]["parameters"]
        props, required = schema["properties"], schema.get("required", [])
        for name in required:
            if name not in kwargs:
                raise ToolContractError(f"{tool}: missing required argument '{name}'")
        for name, val in kwargs.items():
            if name not in props:
                raise ToolContractError(
                    f"{tool}: unknown argument '{name}'; accepts {sorted(props)}")
            spec = props[name]
            expected = _JSON_TYPES.get(spec["type"])
            if expected and not isinstance(val, expected):
                raise ToolContractError(
                    f"{tool}.{name}: expected {spec['type']}, got {type(val).__name__}")
            if spec.get("enum") and val not in spec["enum"]:
                raise ToolContractError(f"{tool}.{name}: {val!r} not in {spec['enum']}")

    def call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        self.validate_input(tool, kwargs)
        started = time.perf_counter()
        result = TOOLS[tool](**kwargs)
        entry = {"step": len(self.trace) + 1, "agent": self.agent, "tool": tool,
                 "reason": reason, "input": _summarise(kwargs), "output": _summarise(result),
                 "ok": "error" not in result,
                 "ms": round((time.perf_counter() - started) * 1000, 2),
                 "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        self.trace.append(entry)
        self.raw_outputs.append(result)
        if "error" in result:
            raise ToolError(f"{tool} failed: {result['error']}")
        return result

    def try_call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        try:
            return self.call(tool, reason, **kwargs)
        except ToolError:
            return dict(TOOLS[tool](**kwargs))


class Connector:
    """One governed surface over every source. No agent touches a source directly."""

    RESOURCES = {
        "department reports":              "read_department_reports",
        "institutional KPIs & dashboards": "read_kpi_definitions",
        "external benchmarks":             "read_external_benchmark",
    }

    def __init__(self, router: ToolRouter):
        self.router = router

    def read(self, resource: str, reason: str, required: bool = True, **params):
        if resource not in self.RESOURCES:
            raise ToolContractError(f"connector exposes no resource '{resource}'; "
                                    f"available: {sorted(self.RESOURCES)}")
        caller = self.router.call if required else self.router.try_call
        return caller(self.RESOURCES[resource], reason, **params)


class Validated:
    def __init__(self, checks: List[tuple], subject: str):
        self.checks, self.subject = checks, subject
        self.failing = [name for name, ok, _ in checks if not ok]
        self.valid = not self.failing

    def __repr__(self):
        return f"<Validated {self.subject}: {'VALID' if self.valid else self.failing}>"


def validate_kpi_result(computed: Dict[str, Any],
                        compared: Optional[Dict[str, Any]] = None) -> Validated:
    ev = computed.get("evidence", [])
    checks = [
        ("value is a finite number",
         isinstance(computed.get("value"), (int, float)) and math.isfinite(computed["value"]),
         f"value={computed.get('value')}"),
        ("the formula used is recorded", bool(computed.get("formula")),
         computed.get("formula", "")),
        ("the figure cites its source system", bool(computed.get("source_system")),
         computed.get("source_system", "")),
        ("raw inputs are recorded", bool(computed.get("inputs_used")),
         str(computed.get("inputs_used", ""))),
        ("every department in the evidence is identified",
         bool(ev) and all(e.get("dept_code") for e in ev), f"{len(ev)} evidence rows"),
        ("evidence covers exactly the departments computed over",
         len(ev) == computed.get("n_departments"), f"{len(ev)} rows"),
        ("provisional figures carry a caveat",
         computed.get("figure_status") == "final" or bool(computed.get("caveat")),
         f"figure_status={computed.get('figure_status')}"),
    ]
    if compared is not None:
        checks.append(("the target names the authority that set it",
                       bool(compared.get("target_source")), compared.get("target_source", "")))
    return Validated(checks, computed.get("kpi_id", "?"))


# =============================================================================


Writing edss_core.py


### Cell 2 · What a skill is here

A skill is not a function with a case baked in. It is a **capability package**:

* **instructions** — the procedure it follows,
* **constraints** — what it must never do,
* **typed input and output schemas** — validated *before* and *after* it runs,
* **a declared tool surface** — and reaching outside it is an error.

All four are enforced at run time, not documented and hoped for. Cell 8 exercises every one.

In [2]:
# =============================================================================
# Cell 2  ·  What a skill is here
# =============================================================================
# Concept 3: a skill is a REUSABLE CAPABILITY PACKAGE — instructions, a typed
# input/output contract, and the tools it is allowed to call — not a function
# with a case baked into it.
#
# The pitfall this lab grades: "Keep skills case-agnostic and typed; hard-coding
# one case breaks reuse." So every skill below is proved against at least two
# different cases with NO code change.

import json, textwrap
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional

from edss_core import (DOMAINS, KPI_DEFINITIONS, REPORTING_PERIOD, ACCESS_REGISTER,
                       ToolRouter, Connector, ToolError, ToolContractError,
                       validate_kpi_result, _typed_rows, DEPARTMENT_REPORTS_CSV)

_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "array": list, "object": dict}


class SkillContractError(Exception):
    """Raised when a skill's input or output violates its declared schema."""


@dataclass
class Skill:
    """A reusable capability package.

    name          stable identifier
    description   what it does, in one line
    instructions  the procedure it follows — the part a human or a model reads
    constraints   what it must never do
    input_schema  typed contract, validated BEFORE the skill runs
    output_schema typed contract, validated AFTER it runs
    tools         the tool names it is permitted to call
    """
    name: str
    description: str
    instructions: str
    constraints: List[str]
    input_schema: Dict[str, Any]
    output_schema: Dict[str, Any]
    tools: List[str]
    _fn: Optional[Callable] = None
    invocations: List[Dict[str, Any]] = field(default_factory=list)

    # ---- schema validation, both directions ----------------------------
    def _validate(self, payload: Dict[str, Any], schema: Dict[str, Any],
                  where: str) -> None:
        props, required = schema.get("properties", {}), schema.get("required", [])
        for name in required:
            if name not in payload:
                raise SkillContractError(
                    f"{self.name} {where}: missing required field '{name}'")
        for name, value in payload.items():
            if name not in props:
                raise SkillContractError(
                    f"{self.name} {where}: unexpected field '{name}'; "
                    f"accepts {sorted(props)}")
            expected = _JSON_TYPES.get(props[name]["type"])
            if expected and not isinstance(value, expected):
                raise SkillContractError(
                    f"{self.name} {where}.{name}: expected {props[name]['type']}, "
                    f"got {type(value).__name__}")
            if props[name].get("enum") and value not in props[name]["enum"]:
                raise SkillContractError(
                    f"{self.name} {where}.{name}: {value!r} not in "
                    f"{props[name]['enum']}")

    def run(self, router: "ToolRouter", **inputs) -> Dict[str, Any]:
        self._validate(inputs, self.input_schema, "input")
        before = len(router.trace)
        result = self._fn(router, **inputs)
        self._validate(result, self.output_schema, "output")
        used = [e["tool"] for e in router.trace[before:]]
        stray = [t for t in used if t not in self.tools]
        if stray:
            raise SkillContractError(
                f"{self.name} called tools outside its declared surface: {stray}")
        self.invocations.append({"inputs": inputs, "tools_used": used,
                                 "output_keys": sorted(result)})
        return result

    def card(self) -> str:
        return (f"SKILL  {self.name}\n"
                f"  {self.description}\n"
                f"  inputs   : {list(self.input_schema.get('properties', {}))}"
                f"  (required: {self.input_schema.get('required', [])})\n"
                f"  outputs  : {list(self.output_schema.get('properties', {}))}\n"
                f"  tools    : {self.tools}\n"
                f"  constraints:\n"
                + "\n".join(f"    - {c}" for c in self.constraints))


SKILL_REGISTRY: Dict[str, Skill] = {}


def register(skill: Skill) -> Skill:
    SKILL_REGISTRY[skill.name] = skill
    return skill


print("Skill framework ready.")
print("A skill declares: instructions, constraints, typed input/output schemas,")
print("and the exact set of tools it may call. All four are enforced at run time.")


Skill framework ready.
A skill declares: instructions, constraints, typed input/output schemas,
and the exact set of tools it may call. All four are enforced at run time.


### Cell 3 · Skill 1 — `review_domain`

Reviews one domain's KPIs against their approved targets. The KPI list is derived from the registry at run time (`KPI_DEFINITIONS`), so the skill has no idea which domains exist until you call it — which is exactly what Cell 9's run-time-KPI test exploits.

Note the constraints: it **reports, it does not advise**. Recommending an action is the Report Generation Agent's job, behind the human gate.

In [3]:
# =============================================================================
# Cell 3  ·  Skill 1 — review_domain
# =============================================================================
# Reviews ONE domain's KPIs against their approved targets. The domain is an
# INPUT, not a branch in the code: the same skill serves Academic, Research,
# Placement and Finance with no change.

REVIEW_DOMAIN = Skill(
    name="review_domain",
    description="Review one domain's approved KPIs against their targets, with "
                "per-department evidence.",
    instructions=(
        "1. Read the department rows for the requested period through the connector.\n"
        "2. If a department filter is supplied, narrow to it and refuse an empty subset.\n"
        "3. For every approved KPI belonging to the domain: read its definition, compute\n"
        "   it with the registered formula, and compare it to its approved target.\n"
        "4. Return the KPI set with variance, status, source system and figure status.\n"
        "5. Report which KPIs are off target; never recommend an action — that is the\n"
        "   Report Generation Agent's job, behind the human gate."),
    constraints=[
        "Use only approved KPI definitions; never invent a formula.",
        "Never average department percentages — aggregate raw counts.",
        "Label any figure that is not final; never present it as an actual.",
        "Emit no recommendation and take no action.",
    ],
    input_schema={
        "type": "object",
        "properties": {
            "domain": {"type": "string", "enum": DOMAINS},
            "period": {"type": "string"},
            "category": {"type": "string", "enum": ["circuit", "core"]},
        },
        "required": ["domain", "period"],
    },
    output_schema={
        "type": "object",
        "properties": {
            "domain": {"type": "string"},
            "period": {"type": "string"},
            "scope": {"type": "string"},
            "kpis": {"type": "array"},
            "off_target": {"type": "array"},
            "headline": {"type": "string"},
            "provisional": {"type": "boolean"},
        },
        "required": ["domain", "period", "scope", "kpis", "off_target", "headline",
                     "provisional"],
    },
    tools=["read_department_reports", "filter_rows", "read_kpi_definitions",
           "compute_kpi", "compare_to_target"],
)


def _review_domain(router, domain, period, category=None):
    connector = Connector(router)
    rows = connector.read("department reports",
                          reason=f"review_domain({domain}): per-department metrics",
                          period=period)["rows"]
    scope = "all departments"
    if category:
        rows = router.call("filter_rows",
                           reason=f"review_domain({domain}): narrow to {category} branches",
                           rows=rows, field="category", value=category)["rows"]
        scope = f"{category} departments"

    kpi_ids = [k for k, d in KPI_DEFINITIONS.items() if d["domain"] == domain]
    kpis, provisional = [], False
    for kpi_id in kpi_ids:
        connector.read("institutional KPIs & dashboards",
                       reason=f"{kpi_id}: approved formula and target",
                       kpi_ids=[kpi_id])
        computed = router.call("compute_kpi", reason=f"{kpi_id}: apply the formula",
                               kpi_id=kpi_id, rows=rows)
        compared = router.call("compare_to_target", reason=f"{kpi_id}: variance vs target",
                               kpi_id=kpi_id, value=computed["value"])
        gate = validate_kpi_result(computed, compared)
        if not gate.valid:
            raise ToolError(f"validation gate failed for {kpi_id}: {gate.failing}")
        provisional = provisional or computed["figure_status"] != "final"
        kpis.append({
            "kpi_id": kpi_id, "name": computed["name"], "value": computed["value"],
            "unit": computed["unit"], "target": compared["target"],
            "target_source": compared["target_source"], "variance": compared["variance"],
            "status": compared["status"], "formula": computed["formula"],
            "source_system": computed["source_system"],
            "figure_status": computed["figure_status"],
            "rounding_warning": compared["rounding_warning"],
            "evidence": computed["evidence"],
        })

    off = [k["kpi_id"] for k in kpis if k["status"] != "on target"]
    headline = (f"{domain}: {len(off)} of {len(kpis)} KPIs off target across {scope} "
                f"for {period}." + (" Figures include a provisional source."
                                    if provisional else ""))
    return {"domain": domain, "period": period, "scope": scope, "kpis": kpis,
            "off_target": off, "headline": headline, "provisional": provisional}


REVIEW_DOMAIN._fn = _review_domain
register(REVIEW_DOMAIN)
print(REVIEW_DOMAIN.card())


SKILL  review_domain
  Review one domain's approved KPIs against their targets, with per-department evidence.
  inputs   : ['domain', 'period', 'category']  (required: ['domain', 'period'])
  outputs  : ['domain', 'period', 'scope', 'kpis', 'off_target', 'headline', 'provisional']
  tools    : ['read_department_reports', 'filter_rows', 'read_kpi_definitions', 'compute_kpi', 'compare_to_target']
  constraints:
    - Use only approved KPI definitions; never invent a formula.
    - Never average department percentages — aggregate raw counts.
    - Label any figure that is not final; never present it as an actual.
    - Emit no recommendation and take no action.


### Cell 4 · Skill 2 — `format_document`

Knows nothing about KPIs, departments or this worklet. It takes a title, metadata and a list of sections, and supports two templates. That ignorance is the point — it is why Cell 6 can hand it a completely unrelated payload.

It declares **zero tools**, and Cell 9 verifies it never called one.

In [4]:
# =============================================================================
# Cell 4  ·  Skill 2 — format_document
# =============================================================================
# Renders ANY structured payload into a leadership-ready document. It knows
# nothing about KPIs: it takes a title, metadata and a list of sections, and
# supports more than one template. That is what makes it reusable — Cell 6
# feeds it a KPI review and a completely unrelated payload.

FORMAT_DOCUMENT = Skill(
    name="format_document",
    description="Render a titled, sectioned payload as a leadership-ready document "
                "in a chosen template.",
    instructions=(
        "1. Emit the title, then the metadata block as key: value lines.\n"
        "2. Emit each section in order: a heading, then its bullets, then its table\n"
        "   if it has one.\n"
        "3. In 'brief' template, number the sections; in 'memo' template, do not.\n"
        "4. Append the source line verbatim if one is supplied.\n"
        "5. Reproduce every value exactly as given — never reformat a number."),
    constraints=[
        "Never compute, round or reword a value; this skill only lays out.",
        "Never drop a section or a caveat that was passed in.",
        "Stay domain-agnostic: no KPI, department or worklet logic here.",
    ],
    input_schema={
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "meta": {"type": "object"},
            "sections": {"type": "array"},
            "template": {"type": "string", "enum": ["brief", "memo"]},
            "source_line": {"type": "string"},
        },
        "required": ["title", "sections"],
    },
    output_schema={
        "type": "object",
        "properties": {"document": {"type": "string"},
                       "template": {"type": "string"},
                       "section_count": {"type": "integer"},
                       "line_count": {"type": "integer"}},
        "required": ["document", "template", "section_count", "line_count"],
    },
    tools=[],          # a pure formatting skill: it calls no tool at all
)


def _format_document(router, title, sections, meta=None, template="brief",
                     source_line=None):
    out = [title, "=" * len(title)]
    for k, v in (meta or {}).items():
        out.append(f"{k}: {v}")
    out.append("")
    for i, sec in enumerate(sections, 1):
        heading = sec.get("heading", "")
        out.append(f"{i}. {heading}" if template == "brief" else heading.upper())
        out.append("-" * 70)
        for b in sec.get("bullets", []):
            out.append(textwrap.fill(f"  - {b}", 88, subsequent_indent="    "))
        table = sec.get("table")
        if table:
            widths = [max(len(str(r[c])) for r in [dict(zip(table["columns"],
                                                            table["columns"]))] + table["rows"])
                      for c in table["columns"]]
            out.append("  " + "  ".join(str(c).ljust(w) for c, w in
                                        zip(table["columns"], widths)))
            for row in table["rows"]:
                out.append("  " + "  ".join(str(row[c]).ljust(w) for c, w in
                                            zip(table["columns"], widths)))
        out.append("")
    if source_line:
        out.append(source_line)
    document = "\n".join(out)
    return {"document": document, "template": template,
            "section_count": len(sections), "line_count": len(document.splitlines())}


FORMAT_DOCUMENT._fn = _format_document
register(FORMAT_DOCUMENT)
print(FORMAT_DOCUMENT.card())
print(f"\nRegistry now holds {len(SKILL_REGISTRY)} skills: {sorted(SKILL_REGISTRY)}")


SKILL  format_document
  Render a titled, sectioned payload as a leadership-ready document in a chosen template.
  inputs   : ['title', 'meta', 'sections', 'template', 'source_line']  (required: ['title', 'sections'])
  outputs  : ['document', 'template', 'section_count', 'line_count']
  tools    : []
  constraints:
    - Never compute, round or reword a value; this skill only lays out.
    - Never drop a section or a caveat that was passed in.
    - Stay domain-agnostic: no KPI, department or worklet logic here.

Registry now holds 2 skills: ['format_document', 'review_domain']


### Cell 5 · Reuse proof A — one skill, five cases

The acceptance line asks for two. This runs all four domains plus a category-filtered subset through the **same skill object**, with zero lines changed between cases.

In [5]:
# =============================================================================
# Cell 5  ·  Reuse proof A — one skill, four domains, zero code changes
# =============================================================================
# The Lab 3 acceptance line: "The same skill works for two different cases
# without code changes." Here it is run against all four, plus a filtered
# subset — five cases, one implementation.

router = ToolRouter(agent="Lab 3 skill runner")
reviews = {}

print("=" * 82)
print("review_domain — same skill object, five different cases")
print("=" * 82)
for domain in DOMAINS:
    reviews[domain] = REVIEW_DOMAIN.run(router, domain=domain, period=REPORTING_PERIOD)
    print(f"\n  {reviews[domain]['headline']}")
    for k in reviews[domain]["kpis"]:
        flag = "  [PROVISIONAL]" if k["figure_status"] != "final" else ""
        print(f"     {k['kpi_id']:<8} {k['name']:<28} {k['value']:>9.2f} "
              f"vs {k['target']:>7.2f}  {k['status']}{flag}")

# Fifth case: same skill, narrowed to the circuit branches.
circuit = REVIEW_DOMAIN.run(router, domain="Placement", period=REPORTING_PERIOD,
                            category="circuit")
print(f"\n  {circuit['headline']}")
for k in circuit["kpis"]:
    print(f"     {k['kpi_id']:<8} {k['name']:<28} {k['value']:>9.2f} "
          f"vs {k['target']:>7.2f}  {k['status']}")

print("\n" + "-" * 82)
print(f"  invocations of this ONE skill object : {len(REVIEW_DOMAIN.invocations)}")
print(f"  distinct cases served               : "
      f"{len({(i['inputs']['domain'], i['inputs'].get('category')) for i in REVIEW_DOMAIN.invocations})}")
print(f"  lines of code changed between cases : 0  (the case is an input, not a branch)")


review_domain — same skill object, five different cases

  Academic: 2 of 2 KPIs off target across all departments for academic year 2025-26.
     KPI-A1   Institution pass percentage      91.85 vs   92.00  at risk
     KPI-A2   CO/PO attainment index            2.48 vs    2.50  at risk

  Research: 3 of 3 KPIs off target across all departments for academic year 2025-26.
     KPI-R1   Publications per faculty          1.19 vs    1.50  below target
     KPI-R2   Citations per faculty             7.26 vs    9.00  below target
     KPI-R3   Grants sanctioned               242.50 vs  300.00  below target

  Placement: 3 of 3 KPIs off target across all departments for academic year 2025-26.
     KPI-P1   Placement percentage             75.12 vs   80.00  below target
     KPI-P2   Average package                   7.67 vs    8.00  below target
     KPI-P3   Internship completion            73.38 vs   85.00  below target

  Finance: 1 of 1 KPIs off target across all departments for academic 

### Cell 6 · Reuse proof B — one formatter, two unrelated payloads

Case 1 is the Research KPI review as a numbered brief. Case 2 is a placement-drive logistics memo — different shape, different template, same skill.

If the formatter had KPI logic inside it, case 2 would be impossible without a code change. That is the pitfall this lab names.

In [6]:
# =============================================================================
# Cell 6  ·  Reuse proof B — one formatter, two unrelated payloads
# =============================================================================
# format_document has no idea what a KPI is. Case 1 is a KPI review; case 2 is
# a placement-drive memo with a completely different shape. Same skill, same
# call signature, two templates.

# ---- case 1: render the Research review as a numbered brief ----------------
research = reviews["Research"]
case1 = FORMAT_DOCUMENT.run(
    router,
    title=f"Research performance review — {research['period']}",
    meta={"Prepared for": "Principal / Registrar", "Scope": research["scope"],
          "Status": "DRAFT — not circulated"},
    template="brief",
    sections=[
        {"heading": "Headline", "bullets": [research["headline"]]},
        {"heading": "KPIs against target",
         "table": {"columns": ["KPI", "Name", "Value", "Target", "Status"],
                   "rows": [{"KPI": k["kpi_id"], "Name": k["name"],
                             "Value": f"{k['value']:.2f}", "Target": f"{k['target']:.2f}",
                             "Status": k["status"]} for k in research["kpis"]]}},
        {"heading": "Off target", "bullets": research["off_target"] or ["none"]},
    ],
    source_line="Source: " + research["kpis"][0]["source_system"])

print("=" * 82)
print("CASE 1 — a KPI review rendered as a brief")
print("=" * 82)
print(case1["document"])

# ---- case 2: a completely different payload, rendered as a memo ------------
case2 = FORMAT_DOCUMENT.run(
    router,
    title="Placement drive — logistics memo",
    meta={"To": "Placement Cell", "From": "Registrar's office"},
    template="memo",
    sections=[
        {"heading": "Arrangements", "bullets": [
            "Two interview halls booked in the CS block for the recruiter panel.",
            "Transport to be arranged from the main gate for visiting recruiters."]},
        {"heading": "Slots",
         "table": {"columns": ["Day", "Session", "Hall"],
                   "rows": [{"Day": "Monday", "Session": "Pre-placement talk", "Hall": "A1"},
                            {"Day": "Tuesday", "Session": "Technical round", "Hall": "A2"}]}},
    ])

print("\n" + "=" * 82)
print("CASE 2 — an unrelated memo, same skill, different template")
print("=" * 82)
print(case2["document"])

print("\n" + "-" * 82)
print(f"  format_document invocations : {len(FORMAT_DOCUMENT.invocations)}")
print(f"  templates exercised         : "
      f"{sorted({i['inputs'].get('template', 'brief') for i in FORMAT_DOCUMENT.invocations})}")
print(f"  tools called by this skill  : "
      f"{sorted({t for i in FORMAT_DOCUMENT.invocations for t in i['tools_used']}) or 'none — it is pure layout'}")


CASE 1 — a KPI review rendered as a brief
Research performance review — academic year 2025-26
Prepared for: Principal / Registrar
Scope: all departments
Status: DRAFT — not circulated

1. Headline
----------------------------------------------------------------------
  - Research: 3 of 3 KPIs off target across all departments for academic year 2025-26.

2. KPIs against target
----------------------------------------------------------------------
  KPI     Name                      Value   Target  Status      
  KPI-R1  Publications per faculty  1.19    1.50    below target
  KPI-R2  Citations per faculty     7.26    9.00    below target
  KPI-R3  Grants sanctioned         242.50  300.00  below target

3. Off target
----------------------------------------------------------------------
  - KPI-R1
  - KPI-R2
  - KPI-R3

Source: Research repository

CASE 2 — an unrelated memo, same skill, different template
Placement drive — logistics memo
To: Placement Cell
From: Registrar's office

ARRA

### Cell 7 · Composition — skills calling skills

The two skills compose into a whole-institution review with a loop and no new domain logic. This is the shape Lab 7's node graph will wire explicitly and Lab 8's swarm will run in parallel.

In [7]:
# =============================================================================
# Cell 7  ·  Composition — skills calling skills
# =============================================================================
# The two skills compose into the institution-wide review with one loop and no
# new logic: review_domain runs per domain, format_document lays the result out.
# This is the shape Lab 7's node graph and Lab 8's swarm will orchestrate.

def institution_review(router, period, domains=DOMAINS):
    """Compose the registry's skills into a whole-institution review."""
    per_domain = {d: SKILL_REGISTRY["review_domain"].run(router, domain=d, period=period)
                  for d in domains}
    all_kpis = [k for r in per_domain.values() for k in r["kpis"]]
    off = [k for k in all_kpis if k["status"] != "on target"]
    provisional = [k for k in all_kpis if k["figure_status"] != "final"]
    rounding = [k for k in all_kpis if k["rounding_warning"]]

    return SKILL_REGISTRY["format_document"].run(
        router,
        title=f"Institution performance review — {period}",
        meta={"Prepared for": "Principal / Registrar / Governing Council",
              "Domains": ", ".join(domains),
              "Status": "DRAFT — not circulated without approval"},
        template="brief",
        sections=[
            {"heading": "Headline", "bullets": [r["headline"] for r in per_domain.values()]},
            {"heading": "All KPIs against target",
             "table": {"columns": ["KPI", "Name", "Domain", "Value", "Target", "Status"],
                       "rows": [{"KPI": k["kpi_id"], "Name": k["name"],
                                 "Domain": KPI_DEFINITIONS[k["kpi_id"]]["domain"],
                                 "Value": f"{k['value']:.2f}", "Target": f"{k['target']:.2f}",
                                 "Status": k["status"]} for k in all_kpis]}},
            {"heading": "Flags", "bullets":
                ([f"{k['kpi_id']} is PROVISIONAL — {k['name']}" for k in provisional]
                 + [f"{k['kpi_id']} would read as on-target if rounded — "
                    f"{k['rounding_warning']}" for k in rounding]) or ["none"]},
        ],
        source_line="Every figure computed by the registered KPI formulas from "
                    "Registrar-authorised department records.")


composed = institution_review(router, REPORTING_PERIOD)
print(composed["document"])
print("-" * 82)
print(f"  skills used          : review_domain x{len(DOMAINS)} -> format_document x1")
print(f"  new domain logic     : none — composition only")
print(f"  tool calls behind it : {len(router.trace)} in total so far")


Institution performance review — academic year 2025-26
Prepared for: Principal / Registrar / Governing Council
Domains: Academic, Research, Placement, Finance
Status: DRAFT — not circulated without approval

1. Headline
----------------------------------------------------------------------
  - Academic: 2 of 2 KPIs off target across all departments for academic year 2025-26.
  - Research: 3 of 3 KPIs off target across all departments for academic year 2025-26.
  - Placement: 3 of 3 KPIs off target across all departments for academic year 2025-26.
  - Finance: 1 of 1 KPIs off target across all departments for academic year 2025-26.
    Figures include a provisional source.

2. All KPIs against target
----------------------------------------------------------------------
  KPI     Name                         Domain     Value   Target  Status      
  KPI-A1  Institution pass percentage  Academic   91.85   92.00   at risk     
  KPI-A2  CO/PO attainment index       Academic   2.48    2.50

### Cell 8 · The contracts are real

Seven violations, all refused: unknown enum value, missing required field, wrong type, unexpected field, and a rogue skill that declares no tools but tries to call one.

A skill that only *claims* to be typed is not typed.

In [8]:
# =============================================================================
# Cell 8  ·  The contracts are real — every guardrail exercised
# =============================================================================
# A skill that only *claims* to be typed is not typed. Each case below must be
# refused.

print("=" * 82)
print("SKILL CONTRACT ENFORCEMENT")
print("=" * 82)

refusals = []
cases = [
    ("input: unknown domain",
     lambda: REVIEW_DOMAIN.run(router, domain="Sustainability", period=REPORTING_PERIOD)),
    ("input: missing a required field",
     lambda: REVIEW_DOMAIN.run(router, domain="Academic")),
    ("input: wrong type",
     lambda: REVIEW_DOMAIN.run(router, domain="Academic", period=2025)),
    ("input: field the skill does not accept",
     lambda: REVIEW_DOMAIN.run(router, domain="Academic", period=REPORTING_PERIOD,
                               sort_by="value")),
    ("input: enum violation on the formatter",
     lambda: FORMAT_DOCUMENT.run(router, title="x", sections=[], template="poster")),
    ("input: formatter missing its sections",
     lambda: FORMAT_DOCUMENT.run(router, title="x")),
]
for label, call in cases:
    try:
        call()
        print(f"  [!!] NOT REFUSED — {label}")
    except SkillContractError as exc:
        refusals.append((label, str(exc)))
        print(f"  [refused] {label}\n             {exc}")

# ---- a skill may not reach for a tool outside its declared surface ---------
ROGUE = Skill(name="rogue", description="declares no tools but tries to call one",
              instructions="-", constraints=["-"],
              input_schema={"type": "object", "properties": {}, "required": []},
              output_schema={"type": "object", "properties": {"ok": {"type": "boolean"}},
                             "required": ["ok"]},
              tools=[])          # <- declares nothing
ROGUE._fn = lambda r: (r.call("read_department_reports", reason="undeclared reach",
                              period=REPORTING_PERIOD), {"ok": True})[1]
try:
    ROGUE.run(router)
    print("  [!!] NOT REFUSED — undeclared tool use")
except SkillContractError as exc:
    refusals.append(("undeclared tool use", str(exc)))
    print(f"  [refused] skill reached outside its declared tool surface\n             {exc}")

# ---- and the domain guardrail: no recommendation may leak out -------------
banned = ["recommend", "should ", "must increase", "we suggest"]
leaks = [w for w in banned
         for r in reviews.values() if w in r["headline"].lower()]
print(f"\n  review_domain emitted a recommendation: {bool(leaks)}  "
      f"(constraint: it reports, it does not advise)")


SKILL CONTRACT ENFORCEMENT
  [refused] input: unknown domain
             review_domain input.domain: 'Sustainability' not in ['Academic', 'Research', 'Placement', 'Finance']
  [refused] input: missing a required field
             review_domain input: missing required field 'period'
  [refused] input: wrong type
             review_domain input.period: expected string, got int
  [refused] input: field the skill does not accept
             review_domain input: unexpected field 'sort_by'; accepts ['category', 'domain', 'period']
  [refused] input: enum violation on the formatter
             format_document input.template: 'poster' not in ['brief', 'memo']
  [refused] input: formatter missing its sections
             format_document input: missing required field 'sections'
  [refused] skill reached outside its declared tool surface
             rogue called tools outside its declared surface: ['read_department_reports']

  review_domain emitted a recommendation: False  (constraint: it

### Cell 9 · Acceptance checks — the rubric, executed

Ten checks. The one that matters most is **C3**: a KPI is added to the registry at run time and the same skill object serves it immediately. That is a stronger proof of case-agnosticism than reading the source, because it demonstrates the behaviour rather than inspecting the text.

In [9]:
# =============================================================================
# Cell 9  ·  Acceptance checks — the rubric, executed
# =============================================================================
# Worklet 15, Lab 3 acceptance: "The same skill works for two different cases
# without code changes."
# Pitfall guarded: "Keep skills case-agnostic and typed; hard-coding one case
# breaks reuse."

checks = []


def check(name, condition, evidence=""):
    checks.append((name, bool(condition), evidence))


cases_served = {(i["inputs"]["domain"], i["inputs"].get("category"))
                for i in REVIEW_DOMAIN.invocations}
check("C1  review_domain served more than one case, unchanged", len(cases_served) >= 2,
      f"{len(cases_served)} distinct cases: {sorted(str(c) for c in cases_served)}")

check("C2  format_document served two unrelated payload shapes",
      len(FORMAT_DOCUMENT.invocations) >= 2
      and len({i["inputs"].get("template", "brief")
               for i in FORMAT_DOCUMENT.invocations}) >= 2,
      f"{len(FORMAT_DOCUMENT.invocations)} invocations across "
      f"{len({i['inputs'].get('template', 'brief') for i in FORMAT_DOCUMENT.invocations})} templates")

# The strongest proof that nothing is hard-coded: add a KPI to the registry at
# RUN TIME and show the same skill object picks it up with no code change.
before_n = len(REVIEW_DOMAIN.run(router, domain="Academic",
                                 period=REPORTING_PERIOD)["kpis"])
KPI_DEFINITIONS["KPI-A3"] = {
    "name": "Curriculum compliance", "domain": "Academic", "method": "ratio",
    "numerator": "students_passed", "denominator": "students_appeared", "scale": 100,
    "unit": "%", "formula": "sum(students_passed) / sum(students_appeared) x 100",
    "target": 95.0, "target_source": "NAAC criterion 1",
    "direction": "higher_is_better", "source_system": "Academic ERP (curriculum module)"}
after = REVIEW_DOMAIN.run(router, domain="Academic", period=REPORTING_PERIOD)
after_n = len(after["kpis"])
picked_up = [k["kpi_id"] for k in after["kpis"]]
del KPI_DEFINITIONS["KPI-A3"]          # restore the registry
check("C3  a KPI added at run time is served with zero code change",
      after_n == before_n + 1 and "KPI-A3" in picked_up,
      f"Academic KPIs {before_n} -> {after_n}; skill returned {picked_up}")

check("C4  every skill declares typed input AND output schemas",
      all(s.input_schema.get("properties") is not None
          and s.output_schema.get("properties") is not None
          for s in SKILL_REGISTRY.values()),
      f"{len(SKILL_REGISTRY)} skills in the registry")

check("C5  malformed input is refused before the skill body runs",
      len(refusals) >= 6,
      f"{len(refusals)} contract violations refused in Cell 8: "
      f"{[r[0] for r in refusals[:3]]}…")

check("C6  a skill cannot call a tool outside its declared surface",
      len(ROGUE.invocations) == 0
      and any("outside its declared surface" in why for _, why in refusals),
      "the rogue skill's undeclared read_department_reports call was refused, so it "
      "recorded zero successful invocations")

check("C7  skills compose without new domain logic",
      composed["section_count"] == 3 and composed["line_count"] > 20,
      f"institution_review = review_domain x{len(DOMAINS)} -> format_document, "
      f"{composed['line_count']} lines out")

check("C8  the formatter stays pure — it calls no tool at all",
      not any(i["tools_used"] for i in FORMAT_DOCUMENT.invocations),
      "zero tool calls across all format_document invocations")

check("C9  every figure in the composed review came from a tool",
      all(k["source_system"] for r in reviews.values() for k in r["kpis"]),
      f"{sum(len(r['kpis']) for r in reviews.values())} KPIs, each citing a source system")

check("C10 provisional and rounding flags survive into the output",
      any(k["figure_status"] != "final" for r in reviews.values() for k in r["kpis"])
      and any(k["rounding_warning"] for r in reviews.values() for k in r["kpis"]),
      "Finance provisional + rounding warnings present in the skill output")

print("=" * 82)
print("LAB 3 ACCEPTANCE CHECKS")
print("=" * 82)
for name, ok, ev in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {name}")
    if ev:
        print(f"          {ev}")
failed = [n for n, ok, _ in checks if not ok]
print("=" * 82)
print(f"  {len(checks) - len(failed)}/{len(checks)} checks passed.")
assert not failed, f"FAILED: {failed}"
print("  Lab 3 acceptance criteria satisfied.")


LAB 3 ACCEPTANCE CHECKS
  [PASS]  C1  review_domain served more than one case, unchanged
          5 distinct cases: ["('Academic', None)", "('Finance', None)", "('Placement', 'circuit')", "('Placement', None)", "('Research', None)"]
  [PASS]  C2  format_document served two unrelated payload shapes
          3 invocations across 2 templates
  [PASS]  C3  a KPI added at run time is served with zero code change
          Academic KPIs 2 -> 3; skill returned ['KPI-A1', 'KPI-A2', 'KPI-A3']
  [PASS]  C4  every skill declares typed input AND output schemas
          2 skills in the registry
  [PASS]  C5  malformed input is refused before the skill body runs
          7 contract violations refused in Cell 8: ['input: unknown domain', 'input: missing a required field', 'input: wrong type']…
  [PASS]  C6  a skill cannot call a tool outside its declared surface
          the rogue skill's undeclared read_department_reports call was refused, so it recorded zero successful invocations
  [PASS]  C7

### Cell 10 · Deliverables

Skill definitions (the cards a reviewer reads), the domain reviews, the composed document, the invocation log, the tool trace, and the acceptance results — all exported as JSON for the review sheet's evidence row.

In [10]:
# =============================================================================
# Cell 10 · Deliverables
# =============================================================================
from pathlib import Path

outdir = Path("lab3_artifacts")
outdir.mkdir(exist_ok=True)

skill_cards = {name: {"description": s.description, "instructions": s.instructions,
                      "constraints": s.constraints, "input_schema": s.input_schema,
                      "output_schema": s.output_schema, "tools": s.tools,
                      "invocations": len(s.invocations)}
               for name, s in SKILL_REGISTRY.items()}

artifacts = {
    "lab3_skill_definitions.json": skill_cards,
    "lab3_domain_reviews.json": reviews,
    "lab3_composed_review.md": composed["document"],
    "lab3_invocation_log.json": {n: s.invocations for n, s in SKILL_REGISTRY.items()},
    "lab3_tool_trace.json": router.trace,
    "lab3_acceptance_checks.json": [{"check": n, "passed": ok, "evidence": e}
                                    for n, ok, e in checks],
}
for fname, obj in artifacts.items():
    p = outdir / fname
    p.write_text(obj if isinstance(obj, str)
                 else json.dumps(obj, indent=2, ensure_ascii=False, default=str))

print("=" * 82)
print("ARTIFACTS WRITTEN")
print("=" * 82)
for fname in artifacts:
    print(f"  {outdir / fname}  ({(outdir / fname).stat().st_size:,} bytes)")

print(f"\n  Skills registered      : {len(SKILL_REGISTRY)}  {sorted(SKILL_REGISTRY)}")
print(f"  Total skill invocations: {sum(len(s.invocations) for s in SKILL_REGISTRY.values())}")
print(f"  Tool calls behind them : {len(router.trace)}")
print(f"  Acceptance checks      : {sum(ok for _, ok, _ in checks)}/{len(checks)} passed")


ARTIFACTS WRITTEN
  lab3_artifacts/lab3_skill_definitions.json  (4,102 bytes)
  lab3_artifacts/lab3_domain_reviews.json  (31,481 bytes)
  lab3_artifacts/lab3_composed_review.md  (2,391 bytes)
  lab3_artifacts/lab3_invocation_log.json  (13,950 bytes)
  lab3_artifacts/lab3_tool_trace.json  (74,872 bytes)
  lab3_artifacts/lab3_acceptance_checks.json  (1,935 bytes)

  Skills registered      : 2  ['format_document', 'review_domain']
  Total skill invocations: 14
  Tool calls behind them : 91
  Acceptance checks      : 10/10 passed


---

## Where Lab 3 leaves the worklet

**Nodes activated:** the Plan node and the Assembly node, now expressed as skills rather than inline code.

**What exists now:** two reusable, schema-typed skills that any later agent can call without knowing how they work inside — and that refuse malformed input, refuse undeclared tools, and refuse to leak a recommendation.

**What Lab 4 adds:** memory. Right now every run starts from nothing. Lab 4 separates the **short-term run state** (the draft being worked on) from the **long-term store** (past briefs, prior-year KPIs), and retrieves from the store by tag and by similarity.

> Lab 4 pseudocode (Chapter 15):
> ```
> state.draft = candidate                            # short-term
> hits  = retrieve(store, tags)                      # long-term
> prior = retrieve(history, embed(candidate), k=5)   # similarity
> ```